# IPAJ Gaming — Análise com Apriori

Notebook de mineração de dados do projeto **IPAJ Gaming** (Iran Pablo e Aloisio Junior).

Etapas: carregamento do dataset simulado → análise exploratória → divisão 80/20 →
itemsets frequentes → regras de associação → avaliação no teste.

Todas as funções vêm de `ml/`, as mesmas usadas pelo backend, para que o notebook
e o sistema publicado produzam exatamente os mesmos resultados.

In [1]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().resolve()
if not (ROOT / "ml").exists():
    ROOT = ROOT.parent  # executando de dentro de notebooks/
sys.path.insert(0, str(ROOT))

import pandas as pd
from ml import config
from ml.preprocess import load_baskets, split, to_baskets
from ml.train_apriori import mine
pd.set_option("display.max_colwidth", 80)

## 1. Dataset e catálogo

In [2]:
catalog = pd.read_csv(config.CATALOG_CSV)
tx = load_baskets(config.TRANSACTIONS_CSV)
names = dict(zip(catalog.product_id, catalog.name))
print(f"Produtos: {len(catalog)} | Transações: {len(tx)}")
tx.head()

Produtos: 50 | Transações: 5000


,transaction_id,timestamp,scenario,n_items,items
0,T00001,2025-01-05 17:34:00,playstation,6,P001;P004;P006;P014;P035;P038
1,T00002,2025-09-21 08:01:00,playstation,7,P001;P004;P005;P014;P018;P035;P038
2,T00003,2025-09-27 13:22:00,setup,6,P029;P030;P031;P033;P034;P038
3,T00004,2025-06-09 22:03:00,setup,2,P032;P034
4,T00005,2025-08-18 03:23:00,pc_build,4,P037;P048;P049;P050


In [3]:
tx["n_items"] = tx["n_items"].astype(int)
print(tx["n_items"].describe().round(2))
tx["n_items"].value_counts().sort_index()

count    5000.00
mean        2.83
std         1.75
min         1.00
25%         1.00
50%         3.00
75%         4.00
max         9.00
Name: n_items, dtype: float64


n_items
1    1605
2     877
3     835
4     740
5     526
6     260
7     116
8      37
9       4
Name: count, dtype: int64

In [4]:
tx["scenario"].value_counts(normalize=True).round(3)

scenario
pc_perifericos    0.211
playstation       0.144
switch            0.133
xbox              0.107
setup             0.101
casual            0.091
pc_build          0.081
streaming         0.079
aleatorio         0.053
Name: proportion, dtype: float64

In [5]:
from collections import Counter
freq = Counter(i for b in to_baskets(tx) for i in b)
top = pd.DataFrame([(names[k], v, v / len(tx)) for k, v in freq.most_common(10)],
                   columns=["produto", "transações", "suporte"])
top.round(3)

,produto,transações,suporte
0,Mouse Logitech G203,663,0.133
1,Mousepad Speed Médio,602,0.120
2,Controle DualSense Extra,494,0.099
3,Cabo HDMI 2.1 8K,484,0.097
4,Headset HyperX Cloud II,453,0.091
5,PlayStation 5 Slim,437,0.087
6,Mario Kart 8 Deluxe (Switch),437,0.087
7,EA Sports FC 25 (PS5),424,0.085
8,Teclado Mecânico Redragon Kumara,414,0.083
9,Fonte 650 W 80 Plus Bronze,380,0.076


## 2. Divisão treino/teste (80/20, semente 42)

In [6]:
train_df, test_df = split(tx)
print(len(train_df), len(test_df))
assert set(train_df.transaction_id).isdisjoint(test_df.transaction_id)

4000 1000


## 3. Apriori (mlxtend)

Parâmetros: suporte mínimo = 0.01, confiança mínima = 0.3,
lift mínimo = 1.2, tamanho máximo = 3.

In [7]:
itemsets, rules = mine(to_baskets(train_df))
print(f"Itemsets frequentes: {len(itemsets)} | Regras: {len(rules)}")
pd.Series([len(i["items"]) for i in itemsets]).value_counts().sort_index().rename("itemsets por tamanho")

Itemsets frequentes: 248 | Regras: 394


1     50
2    105
3     93
Name: itemsets por tamanho, dtype: int64

In [8]:
def label(ids): return " + ".join(names[i] for i in ids)
rules_df = pd.DataFrame([{"se": label(r["antecedents"]), "então": label(r["consequents"]),
                          "suporte": r["support"], "confiança": r["confidence"], "lift": r["lift"]}
                         for r in rules])
rules_df.sort_values("lift", ascending=False).head(15).round(3)

,se,então,suporte,confiança,lift
22,Braço Articulado para Monitor + Cabo HDMI 2.1 8K,"Monitor Gamer 24"" 144 Hz",0.016,0.969,18.820
25,Headset HyperX Cloud II + Mousepad Extended RGB,Mouse Razer DeathAdder Essential,0.012,0.960,16.916
27,Mousepad Speed Médio + Mousepad Extended RGB,Mouse Razer DeathAdder Essential,0.017,0.958,16.877
37,Mouse Logitech G203 + Mousepad Extended RGB,Mouse Razer DeathAdder Essential,0.019,0.939,16.547
122,Mouse Logitech G203 + Mouse Razer DeathAdder Essential,Mousepad Extended RGB,0.019,0.778,16.289
84,Webcam Full HD 1080p + Microfone Condensador USB,"Ring Light 10"" com Tripé",0.026,0.841,16.256
125,Headset HyperX Cloud II + Mouse Razer DeathAdder Essential,Mousepad Extended RGB,0.012,0.774,16.213
130,Mouse Razer DeathAdder Essential + Mousepad Speed Médio,Mousepad Extended RGB,0.017,0.764,16.001
58,Nintendo Switch OLED + Super Mario Bros. Wonder (Switch),Capa Protetora para Switch,0.011,0.882,15.480
61,Nintendo Switch OLED + Base de Carregamento Switch,Capa Protetora para Switch,0.020,0.868,15.230


In [9]:
rules_df.sort_values("confiança", ascending=False).head(10).round(3)

,se,então,suporte,confiança,lift
0,Mario Kart 8 Deluxe (Switch) + Base de Carregamento Switch,Nintendo Switch OLED,0.019,1.000,14.388
1,Switch Pro Controller + Capa Protetora para Switch,Nintendo Switch OLED,0.026,1.000,14.388
2,Capa Protetora para Switch + Base de Carregamento Switch,Nintendo Switch OLED,0.020,1.000,14.388
3,Headset Pulse 3D + SSD NVMe 1 TB para PS5,PlayStation 5 Slim,0.024,1.000,11.331
4,SSD NVMe 1 TB para PS5 + Suporte Vertical para Console,PlayStation 5 Slim,0.014,1.000,11.331
5,Teclado Mecânico Redragon Kumara + Teclado Logitech G413 TKL,Mouse Logitech G203,0.014,1.000,7.246
6,Cartão microSD 128 GB + Capa Protetora para Switch,Nintendo Switch OLED,0.036,0.993,14.289
7,Mario Kart 8 Deluxe (Switch) + Capa Protetora para Switch,Nintendo Switch OLED,0.034,0.993,14.285
8,Controle DualSense Extra + SSD NVMe 1 TB para PS5,PlayStation 5 Slim,0.034,0.993,11.249
9,EA Sports FC 25 (PS5) + Suporte Vertical para Console,PlayStation 5 Slim,0.022,0.989,11.204


In [10]:
rules_df[["suporte", "confiança", "lift"]].describe().round(3)

,suporte,confiança,lift
count,394.000,394.000,394.000
mean,0.025,0.638,8.860
std,0.014,0.207,3.162
min,0.010,0.303,2.251
25%,0.015,0.457,6.625
50%,0.020,0.618,8.880
75%,0.032,0.826,10.772
max,0.108,1.000,18.820


## 4. Recomendação e avaliação

In [11]:
from ml.recommendation_engine import RecommendationEngine
engine = RecommendationEngine(rules, itemsets)
for cart in (["P001"], ["P003"], ["P021"], ["P046", "P049"], ["P050"]):
    recs = engine.recommend(cart, limit=3)
    print(label(cart), "->", [(names[r["product_id"]], round(r["confidence"], 2)) for r in recs] or "sem regra aplicável")

PlayStation 5 Slim -> [('Controle DualSense Extra', 0.72), ('EA Sports FC 25 (PS5)', 0.62), ('Headset Pulse 3D', 0.56)]
Nintendo Switch OLED -> [('Cartão microSD 128 GB', 0.65), ('Mario Kart 8 Deluxe (Switch)', 0.63), ('Capa Protetora para Switch', 0.62)]
Teclado Mecânico Redragon Kumara -> [('Mouse Logitech G203', 0.78), ('Mousepad Speed Médio', 0.75), ('Headset HyperX Cloud II', 0.42)]
Placa de Vídeo RTX 4060 + Gabinete Gamer com Vidro Temperado -> [('Fonte 650 W 80 Plus Bronze', 0.59), ('Memória RAM 16 GB DDR4', 0.46), ('SSD NVMe 1 TB para PC', 0.34)]
Fonte 650 W 80 Plus Bronze -> [('Placa de Vídeo RTX 4060', 0.45), ('Memória RAM 16 GB DDR4', 0.42), ('SSD NVMe 1 TB para PC', 0.38)]


As métricas oficiais vêm de `ml/evaluate_apriori.py` (gravadas em `ml/artifacts/metrics.json`).
Recalculamos aqui a partir do artefato para conferência.

In [12]:
metrics = json.loads(config.METRICS_PATH.read_text(encoding="utf-8"))
ev = metrics["evaluation"]
pd.DataFrame({
    "métrica": ["Hit rate @3 (Apriori)", "Hit rate @3 (popularidade)", "Cobertura de carrinhos",
                "Cobertura de itens", "Precisão @3 (cobertos)"],
    "valor (%)": [ev["hit_rate_at_k_all_carts_pct"], ev["baseline_popularity_hit_rate_at_k_pct"],
                  ev["cart_coverage_pct"], ev["item_coverage_pct"], ev["precision_at_k_covered_pct"]],
})

,métrica,valor (%)
0,Hit rate @3 (Apriori),58.72
1,Hit rate @3 (popularidade),11.92
2,Cobertura de carrinhos,95.53
3,Cobertura de itens,74.00
4,Precisão @3 (cobertos),26.39


In [13]:
pd.DataFrame(ev["rule_quality"]["train"], index=["treino"]).T.join(
    pd.DataFrame(ev["rule_quality"]["test"], index=["teste"]).T).round(4)

,treino,teste
support,0.0249,0.0233
confidence,0.6383,0.6182
lift,8.8599,8.6030


In [14]:
# Conferência: o notebook reproduz exatamente o artefato persistido.
import joblib
assert rules == joblib.load(config.RULES_PATH), "regras divergem do artefato"
print("Regras do notebook idênticas a ml/artifacts/rules.joblib")

Regras do notebook idênticas a ml/artifacts/rules.joblib


## 5. Conclusões e limitações

- As regras com maior lift recuperam as associações planejadas no simulador
  (ex.: Switch → capa/microSD, kit de PC → fonte), e mantêm confiança e lift semelhantes no teste,
  indicando que não há sobreajuste às transações de treino.
- O Apriori supera com folga a linha de base por popularidade no Hit rate @3.
- **Limitação principal:** os dados são simulados; o desempenho mede quão bem o algoritmo
  recupera padrões gerados pelo próprio simulador, não o comportamento de clientes reais.
- Itens raros (abaixo de 1% de suporte) nunca geram regras; nem toda combinação recebe recomendação.